## Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
1. Extract specific portion of the string from refund_reason using regexp_extract function
1. Extract date and time from the refund_timestamp
1. Write transformed data to the Silver schema

In [0]:
df_refunds = spark.table('gizmobox.bronze.py_refunds')
display(df_refunds)

refund_id,payment_id,refund_timestamp,refund_amount,refund_reason
1,66,2025-01-10T11:30:00Z,85.75,Payment Error:Retailer
2,69,2025-01-03T12:40:15Z,120.5,Order Cancelled:Customer
3,72,2025-01-06T14:45:30Z,65.0,Product Returned:Customer
4,73,2025-01-07T16:10:45Z,210.99,Order Cancelled:Customer
5,75,2025-01-09T18:25:00Z,45.2,Payment Error:Retailer
6,80,2025-01-10T09:35:20Z,130.15,Order Cancelled:Customer
7,83,2025-01-12T11:20:40Z,150.0,Product Returned:Customer
8,85,2025-01-14T13:15:30Z,89.99,Order Cancelled:Customer
9,89,2025-01-15T15:00:00Z,78.5,Payment Error:Retailer
10,91,2025-01-17T16:45:15Z,250.75,Product Returned:Customer


### 1. Extract specific portion of the string from refund_reason using split function

In [0]:
from pyspark.sql import functions as F
df_split_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            "refund_timestamp",
            "refund_amount",
            F.split("refund_reason", ":")[0].alias("refund_reason"),
            F.split("refund_reason", ":")[1].alias("refund_source")
        )
)
display(df_split_refunds)

refund_id,payment_id,refund_timestamp,refund_amount,refund_reason,refund_source
1,66,2025-01-10T11:30:00Z,85.75,Payment Error,Retailer
2,69,2025-01-03T12:40:15Z,120.5,Order Cancelled,Customer
3,72,2025-01-06T14:45:30Z,65.0,Product Returned,Customer
4,73,2025-01-07T16:10:45Z,210.99,Order Cancelled,Customer
5,75,2025-01-09T18:25:00Z,45.2,Payment Error,Retailer
6,80,2025-01-10T09:35:20Z,130.15,Order Cancelled,Customer
7,83,2025-01-12T11:20:40Z,150.0,Product Returned,Customer
8,85,2025-01-14T13:15:30Z,89.99,Order Cancelled,Customer
9,89,2025-01-15T15:00:00Z,78.5,Payment Error,Retailer
10,91,2025-01-17T16:45:15Z,250.75,Product Returned,Customer


### 2. Extract specific portion of the string from refund_reason using regexp_extract function

In [0]:
df_transformed_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            F.date_format('refund_timestamp','yyyy-MM-dd').cast('date').alias('refund_date'),
            F.date_format('refund_timestamp','HH-mm-ss').alias('refund_time'),
            "refund_amount",
            F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
            F.regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("refund_source")
        )
)
display(df_transformed_refunds)

refund_id,payment_id,refund_date,refund_time,refund_amount,refund_reason,refund_source
1,66,2025-01-10,11-30-00,85.75,Payment Error,Retailer
2,69,2025-01-03,12-40-15,120.5,Order Cancelled,Customer
3,72,2025-01-06,14-45-30,65.0,Product Returned,Customer
4,73,2025-01-07,16-10-45,210.99,Order Cancelled,Customer
5,75,2025-01-09,18-25-00,45.2,Payment Error,Retailer
6,80,2025-01-10,09-35-20,130.15,Order Cancelled,Customer
7,83,2025-01-12,11-20-40,150.0,Product Returned,Customer
8,85,2025-01-14,13-15-30,89.99,Order Cancelled,Customer
9,89,2025-01-15,15-00-00,78.5,Payment Error,Retailer
10,91,2025-01-17,16-45-15,250.75,Product Returned,Customer


### 3. Write transformed data to the Silver schema  

In [0]:
df_transformed_refunds.writeTo("gizmobox.silver.py_refunds").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_refunds;

refund_id,payment_id,refund_date,refund_time,refund_amount,refund_reason,refund_source
1,66,2025-01-10,11-30-00,85.75,Payment Error,Retailer
2,69,2025-01-03,12-40-15,120.5,Order Cancelled,Customer
3,72,2025-01-06,14-45-30,65.0,Product Returned,Customer
4,73,2025-01-07,16-10-45,210.99,Order Cancelled,Customer
5,75,2025-01-09,18-25-00,45.2,Payment Error,Retailer
6,80,2025-01-10,09-35-20,130.15,Order Cancelled,Customer
7,83,2025-01-12,11-20-40,150.0,Product Returned,Customer
8,85,2025-01-14,13-15-30,89.99,Order Cancelled,Customer
9,89,2025-01-15,15-00-00,78.5,Payment Error,Retailer
10,91,2025-01-17,16-45-15,250.75,Product Returned,Customer
